# AC One — News-Adjusted Fuel Consumption Forecasting

This use case starts **after** an external XGBoost model has produced monthly station forecasts. A cutoff-aware news agent may adjust each point forecast. The CSV stores two paired scales:

- `forecast_minmax` scored against `actual_minmax`
- `forecast_indexed` scored against `actual_indexed`

There is no unsuffixed `forecast` or `actual` column. Both scales are run through the same baseline and agent pipeline so you can see which input produces better MAE/MAPE.

- Stations: `STATION_A`, `STATION_B` (anonymized)
- Horizons: 1, 2, and 3 months
- Primary metrics: MAE and MAPE (lower is better). MAE is **not** comparable across scales; MAPE is.
- Langfuse traces go to project `air-canada-1` via `LANGFUSE_*` keys

The source export stores the target month in `horizon_date`; forecast origin is reconstructed as `horizon_date - month_horizon` months.

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve()
while not (ROOT / "implementations" / "ac_one").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env", override=False)

# Live agent calls use the Vector proxy and cutoff verifier. Keep False for a free Run All.
RUN_AGENT = False
FORCE_REFRESH = False
RUN_TRACE_EVAL = True
RUN_RATIONALE_JUDGE = False  # extra LLM calls; keep False for a free Run All
AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # advanced alternative
FORECAST_SCALES = ("minmax", "indexed")

from ac_one.analysis import (
    improvement_display_formats,
    metric_display_formats,
    metric_summary,
    paired_improvement,
    predictions_to_frame,
)
from ac_one.analyst_agent import build_fuel_adjustment_config, build_fuel_adjustment_predictor
from ac_one.data import build_ac_one_service, load_forecast_data
from ac_one.predictors import ExternalForecastPredictor
from ac_one.specs import build_backtest_specs, load_experiment_spec
from ac_one.trace_eval import (
    LANGFUSE_PROJECT_NAME,
    connect_langfuse,
    evaluate_trace_forecasts,
    trace_ids_from_results,
)
from aieng.forecasting.evaluation import cached_backtest, load_backtest_result


STORE_DIR = ROOT / "data" / "predictions"
langfuse = connect_langfuse()
print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)
print("FORECAST_SCALES =", FORECAST_SCALES)
print(langfuse.message)

## 1. Load and validate the anonymized export

The loader validates required columns, unique forecast keys, consistent actuals across horizons, and positive horizons. Each scale is a first-class experiment input and is never mixed with the other scale's actuals.

In [ ]:
data = load_forecast_data()
service = build_ac_one_service(data)
experiment = load_experiment_spec()

print(f"Rows: {len(data)} | stations: {sorted(data.station.unique())}")
display(
    data[
        [
            "station",
            "region",
            "forecast_origin",
            "horizon_date",
            "month_horizon",
            "forecast_minmax",
            "actual_minmax",
            "forecast_indexed",
            "actual_indexed",
            "model_version",
        ]
    ].head(9)
)

## 2. Run the external-model baseline on both scales

Each station × horizon is a standard `BacktestSpec` with six explicit origins. The adapter returns the scale-specific XGBoost value. Cache keys include the scale so minmax and indexed artifacts cannot overwrite each other.

In [ ]:
baseline_by_scale = {}
specs_by_scale = {}
for scale in FORECAST_SCALES:
    specs = build_backtest_specs(experiment, data, forecast_scale=scale)
    specs_by_scale[scale] = specs
    baseline = ExternalForecastPredictor(data, forecast_scale=scale)
    case_results = {}
    for case_name, spec in specs.items():
        case_results[case_name] = cached_backtest(
            predictor=baseline,
            spec=spec,
            spec_id=f"{experiment.spec_id}_{scale}_{case_name}",
            data_service=service,
            store_dir=STORE_DIR,
            force_refresh=FORCE_REFRESH,
        )
    baseline_by_scale[scale] = case_results
    n = sum(len(result.predictions) for result in case_results.values())
    print(f"{scale}: scored {n} baseline forecasts.")

## 3. Run or load the news-adjusted agent on both scales

The agent sees only the issue date, target month, anonymized station/region, horizon, forecast scale, and that scale's external model forecast. Search is fenced to the issue date. Adjustments are capped at ±20%.

A full run makes 36 analyst calls per scale plus search/verifier calls. Set `RUN_AGENT = True` deliberately. Completed cases are cached independently. Traces are tagged for Langfuse project `air-canada-1`.

In [ ]:
agent_by_scale = {}
if RUN_AGENT and not langfuse.ok:
    print("Agent tracing will not reach Langfuse until credentials for", LANGFUSE_PROJECT_NAME, "are valid.")

for scale in FORECAST_SCALES:
    specs = specs_by_scale[scale]
    agent_predictor_id = f"agent_predictor_ac_one_fuel_news_adjuster_{scale}_{AGENT_MODEL}_continuous"
    case_results = {}
    if RUN_AGENT:
        config = build_fuel_adjustment_config(model=AGENT_MODEL, forecast_scale=scale)
        agent = build_fuel_adjustment_predictor(
            data,
            forecast_scale=scale,
            config=config,
            enable_langfuse_tracing=langfuse.ok,
        )
        agent_predictor_id = agent.predictor_id
        for case_name, spec in specs.items():
            case_results[case_name] = cached_backtest(
                predictor=agent,
                spec=spec,
                spec_id=f"{experiment.spec_id}_{scale}_{case_name}",
                data_service=service,
                store_dir=STORE_DIR,
                force_refresh=FORCE_REFRESH,
            )
    else:
        for case_name in specs:
            cached = load_backtest_result(
                f"{experiment.spec_id}_{scale}_{case_name}",
                agent_predictor_id,
                store_dir=STORE_DIR,
            )
            if cached is not None:
                case_results[case_name] = cached
    agent_by_scale[scale] = case_results
    print(f"{scale}: loaded {len(case_results)}/{len(specs)} agent cases ({agent_predictor_id}).")

if not any(agent_by_scale.values()):
    print("No cached agent results yet. Set RUN_AGENT = True to generate the comparison.")

## 4. Compare MAE and MAPE

Each scale is scored against its matching actual. MAE units differ (`minmax` is roughly 0–1; `indexed` is larger), so do not rank scales by MAE. Rank scales by MAPE, then look at paired agent-vs-baseline improvement within each scale.

A mean improvement of exactly 0 means the agent left the baseline unchanged for those cases — that is a real result, not a missing actual.

In [ ]:
scored_frames = []
for scale in FORECAST_SCALES:
    results = {"External XGBoost": baseline_by_scale[scale]}
    agent_results = agent_by_scale[scale]
    if len(agent_results) == len(specs_by_scale[scale]):
        results["News-adjusted agent"] = agent_results
    scored_frames.append(predictions_to_frame(results, data, forecast_scale=scale))

scored = pd.concat(scored_frames, ignore_index=True) if scored_frames else pd.DataFrame()
overall = metric_summary(scored)
print("Overall (MAE is scale-specific; compare MAPE across scales)")
display(overall.style.format(metric_display_formats()))
display(metric_summary(scored, by=["horizon"]).style.format(metric_display_formats()))
display(metric_summary(scored, by=["station"]).style.format(metric_display_formats()))

if set(overall["predictor"]) >= {"External XGBoost"}:
    baseline_mape = overall[overall["predictor"] == "External XGBoost"].nsmallest(1, "mape_pct")
    print(
        "Lowest-MAPE external scale:",
        baseline_mape["forecast_scale"].iloc[0],
        f"MAPE={baseline_mape['mape_pct'].iloc[0]:.2f}%",
    )

In [ ]:
if "News-adjusted agent" in set(scored.get("predictor", [])):
    display(paired_improvement(scored).style.format(improvement_display_formats()))
    display(paired_improvement(scored, by=["horizon"]).style.format(improvement_display_formats()))
else:
    print("Agent comparison becomes available after all six cached cases exist for a scale.")

## 5. Audit adjustments, rationales, and Langfuse traces

Inspect adjustment size, rationale, evidence, sources, and the Langfuse URL in project `air-canada-1`. Large or frequently one-directional changes are warning signs, especially with only 36 forecast cases per scale.

In [ ]:
agent_audit = scored[scored["predictor"] == "News-adjusted agent"][
    [
        "forecast_scale",
        "station",
        "origin",
        "target_month",
        "horizon",
        "forecast",
        "actual",
        "adjustment_pct",
        "absolute_error",
        "rationale",
        "evidence_summary",
        "source_urls",
        "langfuse_trace_url",
    ]
]
display(agent_audit) if not agent_audit.empty else print("No agent forecasts loaded.")

## 6. Push trace scores to Langfuse

`RUN_TRACE_EVAL = True` fetches each agent trace, joins the matching `actual_*` column, and writes deterministic scores (`absolute_error`, `ape_pct`, improvements, `adjustment_pct`) back to `air-canada-1`.

`RUN_RATIONALE_JUDGE = True` adds extra LLM-as-judge calls for rationale quality. Leave it false for a free Run All.

In [ ]:
if RUN_TRACE_EVAL:
    if not langfuse.ok:
        print(langfuse.message)
    else:
        trace_ids = []
        for scale in FORECAST_SCALES:
            trace_ids.extend(trace_ids_from_results(agent_by_scale[scale]))
        if not trace_ids:
            print("No Langfuse trace ids on cached agent predictions yet. Re-run with RUN_AGENT = True.")
        else:
            trace_scores = evaluate_trace_forecasts(
                trace_ids,
                data,
                push_to_langfuse=True,
                run_judge=RUN_RATIONALE_JUDGE,
                client=langfuse.client,
            )
            print(f"Scored {len(trace_scores)} traces in project {LANGFUSE_PROJECT_NAME}.")
            display(trace_scores)
else:
    print("Trace evaluation skipped. Set RUN_TRACE_EVAL = True after agent traces exist.")

## Interpretation limits

- This anonymized sample has only 36 station/target/horizon rows per scale, so differences are descriptive rather than statistically decisive.
- The station labels do not reveal airports or route networks; the agent must not pretend otherwise.
- News adjustment should be judged against the external model on the **same** scale, not against a naive time-series baseline.
- Re-running the same historical window while tuning prompts turns it into development data. Reserve newer months for a protected evaluation before operational use.